# 03. NumPy

Неделя 3.

**Правило блокнота:** в задачах 3–6 циклы `for`/`while` по элементам массива запрещены. Нужно выражать вычисления операциями над целыми массивами.

**Как работать с блокнотом**
- Сначала решаешь сам, без ИИ. В задачах «предскажи» сначала пишешь ответ в комментарий, потом запускаешь.
- Непонятное поведение — прогони фрагмент в [pythontutor.com](https://pythontutor.com/) и посмотри на стрелки «имя → объект».
- Ячейки Jupyter можно запускать в любом порядке, и состояние (переменные) копится между ними. Это источник «у меня работало». После всех выполненных заданий: **Kernel → Restart & Run All** — блокнот должен пройти сверху вниз без ошибок.

In [ ]:
import numpy as np
import time
rng = np.random.default_rng(0)   # генератор случайных чисел с фиксированным seed: результаты воспроизводимы
print(np.__version__)

## 1. Массив — это блок памяти и его описание

Список Python хранит **ссылки** на отдельные объекты, разбросанные по памяти, и каждый элемент может быть любого типа. `ndarray` устроен иначе:

- **буфер** — один непрерывный кусок памяти с «сырыми» числами одного типа, без объектов Python;
- **`dtype`** — тип элементов (`int32`, `float64`, …), а значит, и их размер в байтах (`itemsize`);
- **`shape`** — размеры по осям;
- **`strides`** — на сколько **байт** сдвинуться в буфере, чтобы перейти к следующему элементу по каждой оси.

Для матрицы 3×4 из `int32` (4 байта) `strides == (16, 4)`: чтобы перейти на следующую строку, нужно пропустить 4 элемента × 4 байта = 16 байт; на следующий столбец — 4 байта.

Именно поэтому многие операции «бесплатны»: срез, транспонирование, `reshape` часто создают только **новое описание** (shape + strides) того же буфера. Такой массив называется **view** (представление). Изменение через view меняет исходный массив.

In [ ]:
a = np.arange(12, dtype=np.int32).reshape(3, 4)
print(a)
print("dtype:", a.dtype, "| shape:", a.shape, "| strides:", a.strides,
      "| itemsize:", a.itemsize, "| nbytes:", a.nbytes)

v = a[:, 1]                 # второй столбец — view
print("strides столбца:", v.strides)   # шаг 16 байт: элементы столбца лежат не подряд
v[:] = -1                   # запись через view
print(a)
print("общая память:", np.shares_memory(a, v))

### Когда view, а когда копия

| Операция | Результат |
|---|---|
| Базовый срез `x[2:5]`, `x[::2]`, `a[:, 1]` | **view** |
| `x.T`, `x.reshape(...)` (если данные можно описать новыми strides) | **view** |
| `x.reshape(...)`, если новыми strides описать нельзя | **копия** |
| Индексация списком индексов `x[[1, 2]]` (fancy indexing) | **копия** |
| Булева маска `x[x > 0]` | **копия** |
| `x.copy()` | **копия** |

Проверка: `np.shares_memory(a, b)` — точный ответ, делят ли массивы память.

**Тонкость:** fancy indexing и маски при **чтении** дают копию, но при **присваивании** (`x[x > 0] = 0`, `x[[1, 2]] = 7`) меняют исходный массив на месте.

### Задача 1. View или копия?

Для каждого выражения запиши прогноз `view`/`copy` в комментарий, потом запусти. Для каждого несовпадения объясни, почему так, через strides: можно ли описать результат как «старый буфер + новые shape и strides»?

In [ ]:
x = np.arange(10)
cases = {
    "x[2:5]": x[2:5],
    "x[[2, 3, 4]]": x[[2, 3, 4]],
    "x[x > 5]": x[x > 5],
    "x.reshape(2, 5)": x.reshape(2, 5),
    "x.reshape(2, 5).T": x.reshape(2, 5).T,
    "x[::2]": x[::2],
    "x.reshape(2, 5).T.reshape(10)": x.reshape(2, 5).T.reshape(10),
}
# Мой прогноз (view/copy для каждого):
#
for k, arr in cases.items():
    print(f"{k:32} ->", "view" if np.shares_memory(x, arr) else "copy")

In [ ]:
# Чтение по маске — копия, присваивание по маске — изменение на месте
y = np.arange(6)
picked = y[y % 2 == 0]
picked[:] = 100
print(y)            # не изменился: picked — копия
y[y % 2 == 0] = 100
print(y)            # изменился: присваивание по маске работает на месте

## 2. Новый массив или изменение на месте

Модель «имя → объект» из блокнота 01 работает и здесь:
- `a = a + 1` создаёт **новый** массив и перепривязывает имя `a`. Другие имена и view продолжают смотреть на старый буфер;
- `a += 1` меняет **существующий** буфер на месте. Изменение видно через все view.

**Тонкости с типами:**
- при `+=` результат пишется в буфер **того же `dtype`**: `int_array += 0.5` бросит `UFuncTypeError`, потому что дробный результат некуда записать;
- целые NumPy фиксированного размера **молча переполняются** (wrap-around), в отличие от `int` Python, у которого нет предела;
- деление `/` всегда даёт `float64`, даже для целых массивов; целочисленное деление — `//`.

In [ ]:
a = np.zeros(3)
view = a[:]
a = a + 1                # новый массив, имя a перепривязано
print(a, view)           # view смотрит на старый буфер: [0. 0. 0.]

a = np.zeros(3)
view = a[:]
a += 1                   # изменение на месте
print(a, view)           # [1. 1. 1.] [1. 1. 1.]

ints = np.array([1, 2, 3])
try:
    ints += 0.5
except Exception as e:
    print(type(e).__name__, ":", e)

small = np.array([100, 120], dtype=np.int8)   # int8: от -128 до 127
print(small * 2)                              # молчаливое переполнение
print(np.array([1, 2]) / 2, np.array([7, -7]) // 2)

### Задача 2. Цена цикла

Сравни время суммы квадратов для 10⁶ элементов: цикл Python против `np.sum(x**2)`.

Объясни разницу. Что делает цикл Python на каждой итерации:
- достаёт из буфера число и **создаёт объект** `float` Python (boxing);
- интерпретатор разбирает операции `*` и `+=` для этого объекта, проверяя типы;
- результат — снова новый объект.

NumPy выполняет весь цикл в скомпилированном коде на C над сырым буфером, без объектов Python, и может использовать векторные инструкции процессора (SIMD).

**Тонкости замера:**
- `time.perf_counter()` — правильные часы для замеров (монотонные, высокого разрешения), а не `time.time()`;
- один замер шумит: запусти несколько раз; в Jupyter удобно `%timeit np.sum(x**2)`;
- `x**2` создаёт временный массив на 8 МБ. Сравни с `np.dot(x, x)`, которое считает то же самое без временного массива.

In [ ]:
x = rng.random(1_000_000)

t0 = time.perf_counter()
s_loop = 0.0
for v in x:
    s_loop += v * v
t1 = time.perf_counter()
s_np = None  # ваш код
t2 = time.perf_counter()

assert np.isclose(s_loop, s_np)
print(f"цикл: {t1 - t0:.3f} c, numpy: {t2 - t1:.4f} c, ускорение x{(t1 - t0) / (t2 - t1):.0f}")

## 3. Оси и broadcasting

**Оси.** `m.mean(axis=0)` усредняет **вдоль** оси 0, то есть «схлопывает» строки и даёт по одному значению на столбец. Для матрицы `(100, 4)` результат имеет форму `(4,)`. `keepdims=True` оставляет схлопнутую ось длиной 1: форма `(1, 4)`.

**Broadcasting** — правила, по которым NumPy выполняет операции над массивами разной формы без копирования данных:
1. формы выравниваются **по правому краю**; недостающие слева оси считаются длиной 1;
2. по каждой оси длины должны быть равны, **или одна из них равна 1**;
3. ось длины 1 «растягивается» до длины другой (без реального копирования, через stride = 0).

Примеры:
- `(100, 4)` и `(4,)` → `(100, 4)`: вектор применяется к каждой строке;
- `(3, 1)` и `(4,)` → `(3, 4)`: «таблица» всех сочетаний;
- `(3,)` и `(4,)` → **ошибка**: 3 ≠ 4 и ни одна не равна 1.

**Тонкость:** `np.std` по умолчанию делит на `n` (`ddof=0`), а pandas и многие статистические пакеты — на `n − 1` (`ddof=1`). Результаты отличаются, и это частый источник «расхождений с Excel».

In [ ]:
col = np.arange(3).reshape(3, 1)   # форма (3, 1)
row = np.arange(4)                  # форма (4,)
print((col + row).shape)
print(col + row)
try:
    np.arange(3) + np.arange(4)
except ValueError as e:
    print("ValueError:", e)

m = np.arange(6.0).reshape(2, 3)
print(m.mean(axis=0), m.mean(axis=0, keepdims=True).shape)
print(np.std([1.0, 2, 3, 4]), np.std([1.0, 2, 3, 4], ddof=1))

### Задача 3. Z-нормировка столбцов

`zscore(m)`: из каждого столбца вычесть его среднее и разделить на его стандартное отклонение (`ddof=0`). Используй `axis` и broadcasting.

Распиши в комментарии формы всех промежуточных массивов: что с чем складывается и по какому правилу broadcasting это работает. Входная матрица не должна измениться.

In [ ]:
def zscore(m):
    raise NotImplementedError

m = rng.normal(5, 3, size=(100, 4))
m_before = m.copy()
z = zscore(m)
assert z.shape == m.shape
assert np.allclose(z.mean(axis=0), 0) and np.allclose(z.std(axis=0), 1)
assert np.array_equal(m, m_before), "zscore изменила входную матрицу"
print("OK")

### Задача 4. Попарные расстояния

`pairwise(p)`: для массива точек `p` формы `(n, d)` вернуть матрицу `(n, n)`, где элемент `[i, j]` — евклидово расстояние между точками `i` и `j`.

Только broadcasting: `p[:, None, :] - p[None, :, :]`. Здесь `None` (то же, что `np.newaxis`) вставляет новую ось длины 1. Распиши формы: `(n, 1, d)` минус `(1, n, d)` → `(n, n, d)`, затем сумма квадратов по последней оси.

**Вопрос:** сколько памяти займёт промежуточный массив `(n, n, d)` из `float64` при n = 10⁴, d = 3? Что будет на ноутбуке с 8 ГБ памяти?

In [ ]:
def pairwise(p):
    raise NotImplementedError

p = np.array([[0.0, 0.0], [3.0, 4.0], [6.0, 8.0]])
d = pairwise(p)
assert d.shape == (3, 3)
assert np.allclose(d, [[0, 5, 10], [5, 0, 5], [10, 5, 0]])
assert np.allclose(d, d.T), "матрица расстояний должна быть симметричной"
print("OK")

### Задача 5. Скользящее среднее через `cumsum`

`moving_avg(x, k)` — массив длины `len(x) − k + 1`, где элемент `i` — среднее окна `x[i : i+k]`. Сложность O(n), без `np.convolve`.

**Идея:** если `c` — префиксные суммы с ведущим нулём (`c[0] = 0`, `c[i] = x[0] + … + x[i−1]`), то сумма окна `x[i : i+k]` равна `c[i+k] − c[i]`. Ведущий ноль можно добавить через `np.concatenate(([0.0], np.cumsum(x)))`.

**Тонкость:** для очень длинных массивов префиксные суммы растут, и разность двух больших близких чисел теряет точность. В итоговом C++-проекте вы будете бороться именно с этой проблемой.

In [ ]:
def moving_avg(x, k):
    raise NotImplementedError

assert np.allclose(moving_avg(np.array([1, 2, 3, 4, 5.0]), 2), [1.5, 2.5, 3.5, 4.5])
assert np.allclose(moving_avg(np.arange(10.0), 10), [4.5])
assert np.allclose(moving_avg(np.arange(5.0), 1), np.arange(5.0))
r = rng.random(1000)
assert np.allclose(moving_avg(r, 7), np.convolve(r, np.ones(7) / 7, mode="valid"))
print("OK")

## 4. Булевы маски и NaN

Сравнение массива даёт **булев массив** той же формы. Его можно использовать как маску для выборки и присваивания.

**Тонкости:**
- для комбинации условий используются `&` (и), `|` (или), `~` (не), а **не** `and`/`or`/`not` — последние работают только с одиночными `True`/`False`;
- у `&` и `|` приоритет **выше**, чем у сравнений, поэтому каждое условие нужно брать в скобки: `(x > 0) & (x < 5)`. Без скобок `x > 0 & x < 5` разбирается как `x > (0 & x) < 5` — это не то, что ты имел в виду;
- `np.nan` существует только для вещественных типов: в целочисленный массив NaN записать нельзя;
- `np.nan != np.nan` — NaN не равен даже самому себе. Проверять только через `np.isnan`;
- `np.mean` массива с NaN даёт NaN; `np.nanmean` пропускает NaN.

In [ ]:
x = np.array([-2, 0, 3, 7, 10])
print((x > 0) & (x < 8))
print(x[(x > 0) & (x < 8)])
try:
    print(x > 0 and x < 8)
except ValueError as e:
    print("ValueError:", e)

print(np.nan == np.nan, np.isnan(np.nan))
f = np.array([1.0, np.nan, 3.0])
print(np.mean(f), np.nanmean(f))

### Задача 6. Очистка выбросов

`clean_outliers(x)` возвращает кортеж `(очищенный массив, число выбросов)`:
- выброс — элемент, для которого `|x − mean| > 3·std` (среднее и std — по всему исходному массиву);
- в очищенном массиве выбросы заменены на `np.nan`;
- очищенный массив — **новый** массив `float64`; исходный не изменяется (вспомни, чем `y = x` отличается от `y = x.astype(float)` или `x.copy()`);
- функция должна работать и для целочисленного входа.

**Вопрос:** сами выбросы увеличивают `std`, по которому их ищут. Почему из-за этого сильный выброс может «спрятать» более слабый? Какие есть более устойчивые критерии (подсказка: медиана)?

In [ ]:
def clean_outliers(x):
    raise NotImplementedError

x = np.r_[rng.normal(0, 1, 1000), [50.0, -40.0]]
x_before = x.copy()
cleaned, n_out = clean_outliers(x)
assert n_out == 2 and np.isnan(cleaned).sum() == 2
assert np.array_equal(x, x_before), "исходный массив изменился"
ints = np.r_[np.zeros(100, dtype=int), [1000]]
c2, n2 = clean_outliers(ints)
assert n2 == 1 and c2.dtype == np.float64
print("OK, среднее без выбросов:", np.nanmean(cleaned))

## Вопросы
1. Что такое `strides` и почему транспонирование «бесплатное»? Чему равны strides у `a.T` для матрицы 3×4 из `int32`?
2. Почему `x[x > 5]` — копия, а `x[2:5]` — view? Почему `x[x > 5] = 0` всё же меняет `x`?
3. Чем `a = a + 1` отличается от `a += 1`, если на `a` есть view?
4. Правила broadcasting: совместимы ли `(3, 1)` и `(4,)`, `(2, 3)` и `(3, 2)`? Какая получится форма?
5. Что вернёт `np.array([1, 2]) / 2`? А `np.array([100], dtype=np.int8) * 2`? Почему?
6. Чем `float32` отличается от `float64` по памяти и точности? Сколько значащих десятичных цифр у каждого?
7. Почему для условий в масках нужны `&` и скобки, а не `and`?
8. Чем `np.std(x)` отличается от `np.std(x, ddof=1)`?